In [0]:
from pyspark.sql import functions as F

In [0]:
# 1. Carregar as fontes tratadas
df_orders = spark.table("bronze_olist_orders_dataset") \
    .withColumn(
        "order_purchase_timestamp",
        F.to_timestamp(
            "order_purchase_timestamp",
            "yyyy-MM-dd HH:mm:ss"
        )
    )

df_reviews = spark.table("bronze_olist_order_reviews_dataset")

df_items = spark.table("bronze_olist_order_items_dataset")

df_customers = spark.table("bronze_olist_customers_dataset")

In [0]:
# 2. Criar a variável alvo solicitada pelo negócio: 1 se review >= 4, senão 0
df_reviews_prepared = (
    df_reviews
    .select(
        "order_id",
        F.col("review_score").cast("int").alias("review_score")
    )
    .withColumn(
        "target_high_review",
        F.when(F.col("review_score") >= 4, 1)
         .otherwise(0)
    )
    .dropDuplicates(["order_id"])
)


In [0]:
# 3. Agregar itens do pedido 
df_items_agg = df_items.groupBy("order_id").agg(
    F.sum("price").alias("total_price"),
    F.sum("freight_value").alias("total_freight"),
    F.count("order_item_id").alias("item_count")
)

In [0]:
# 4. Consolidar a tabela Gold (Feature Store / Model Ready)
df_gold = (
    df_orders
    .join(df_customers, "customer_id", "inner")
    .join(df_items_agg, "order_id", "inner")
    .join(df_reviews_prepared, "order_id", "inner")
    .select(
        "order_id",
        "customer_id",
        "customer_state",
        "customer_city",
        "order_status",
        "order_purchase_timestamp",
        "total_price",
        "total_freight",
        "item_count",
        "review_score",
        "target_high_review"
    )
)

In [0]:
# Visualizar o resultado pronto para o modelo preditivo
display(df_gold.limit(10))

In [0]:
# Gravar a tabela Delta Gold final
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("gold_olist_model_ready")

print("Tabela Gold criada com sucesso!")

## As células abaixo foram criadas para investigar, pois durante o processamento da camada Gold, foi identificado um erro de conversão de tipos na coluna review_score, ocasionado pela presença de valores de data e texto em uma coluna que deveria conter notas de 1 a 5.

In [0]:
#teste
df_reviews = spark.table("bronze_olist_order_reviews_dataset")

df_reviews.printSchema(10)

In [0]:
#teste
display(
    df_reviews.select("review_score").limit(20)
)

In [0]:
#teste
display(
    df_reviews
    .filter(~F.col("review_score").cast("string").rlike("^[1-5]$"))
    .select("review_score")
    .limit(20)
)

In [0]:
#teste
df_reviews.printSchema()

In [0]:
display(
    df_reviews.limit(10)
)

In [0]:
from pyspark.sql import functions as F

df_invalid_reviews = (
    df_reviews
    .filter(
        ~F.col("review_score").cast("string").rlike("^[1-5]$")
    )
)

display(df_invalid_reviews)